# CKD Notebook 2: Model Development and Evaluation

## Project
Trustworthy Agentic AI for Healthcare

## Objective
Develop and evaluate machine learning models for the NHANES-derived,
cross-sectional kidney abnormality research indicator defined in Notebook 1.

## Workflow
1. Load the prepared dataset and verify the target and predictors.
2. Recreate the reproducible training and test split.
3. Train and compare suitable classification models.
4. Evaluate discrimination, precision, recall, F1-score, and calibration.
5. Select decision thresholds using training data only.
6. Evaluate the final selected model on the held-out test set.
7. Save only the research outputs needed for the GitHub repository.

## Research limitations
The target is a cross-sectional laboratory-derived kidney abnormality
indicator. It is not a confirmed clinical diagnosis of CKD and does not
predict future CKD.

The random seed is 42. All preprocessing must be fitted using training
data only, with the held-out test set reserved for final evaluation.

In [2]:
# CKD_NB2_Model_Development_Evaluation.ipynb
# Cell 1: Imports and reproducibility configuration

import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings("default")

print("CKD Notebook 2: Model Development and Evaluation")
print(f"Random seed: {SEED}")
print(f"Python working directory: {Path.cwd()}")
print(f"Python version: {__import__('sys').version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

CKD Notebook 2: Model Development and Evaluation
Random seed: 42
Python working directory: /content
Python version: 3.13.16
Pandas version: 2.2.3
NumPy version: 2.1.3


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
import requests

REPO_API = (
    "https://api.github.com/repos/"
    "MeghaKA/Trustworthy-Agentic-AI-for-Healthcare/contents"
)

response = requests.get(REPO_API, timeout=30)

print("GitHub repository status:", response.status_code)

if response.status_code == 200:
    print("\nRepository root contents:")
    for item in response.json():
        print(f"{item['type']:6} {item['path']}")
else:
    print(response.text[:500])
    print(response.text[:500])

GitHub repository status: 200

Repository root contents:
file   .gitignore
file   LICENSE
file   README.md
dir    data
dir    demonstrator
dir    images
dir    notebooks
file   requirements.txt
dir    results


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
# Cell 3: Inspect existing CKD project files on GitHub

for folder in ["data", "notebooks", "results"]:
    url = (
        "https://api.github.com/repos/"
        "MeghaKA/Trustworthy-Agentic-AI-for-Healthcare/contents/"
        + folder
    )

    r = requests.get(url, timeout=30)
    print(f"\n--- {folder}/ (HTTP {r.status_code}) ---")

    if r.status_code == 200:
        for item in r.json():
            print(f"{item['type']:6} {item['path']}")
    else:
        print(r.text[:300])


--- data/ (HTTP 200) ---
dir    data/processed
dir    data/raw

--- notebooks/ (HTTP 200) ---
file   notebooks/02_Clinical_Cohort_Target_Definition_Feature_Engineering.ipynb
file   notebooks/03_predictive_modeling_and_clinical_evaluation.ipynb
file   notebooks/04_explainable_ai_and_patient_level_interpretation.ipynb
file   notebooks/Notebook1_Data_Integration_and_Preprocessing.ipynb
file   notebooks/Notebook_5_Trust_Fairness_Safety_Evaluation.ipynb
file   notebooks/Notebook_6_Explainability_Consistency_Framework.ipynb
file   notebooks/Notebook_7_Clinical_Decision_Support_Prototype.ipynb
file   notebooks/Notebook_8_Trustworthy_Agentic_Healthcare_Workflow.ipynb
file   notebooks/Notebook_9_Automated_Trustworthy_Healthcare_AI_Demonstrator.ipynb
dir    notebooks/ckd
file   notebooks/readme.md

--- results/ (HTTP 200) ---
dir    results/notebook4_explanation_consistency
dir    results/notebook5_trust_fairness_safety
dir    results/notebook7_cds_prototype
dir    results/notebook8_agent_workf

In [5]:
# Cell 4: Verify available CKD datasets and notebooks

folders_to_check = [
    "data/processed",
    "data/raw",
    "notebooks/ckd",
]

for folder in folders_to_check:
    url = (
        "https://api.github.com/repos/"
        "MeghaKA/Trustworthy-Agentic-AI-for-Healthcare/contents/"
        + folder
    )
    r = requests.get(url, timeout=30)

    print(f"\n--- {folder}/ ---")

    if r.status_code == 200:
        items = r.json()

        if not items:
            print("(empty folder)")
        else:
            for item in items:
                size = item.get("size", 0)
                print(
                    f"{item['type']:6} "
                    f"{item['path']} "
                    f"({size:,} bytes)"
                )
    else:
        print(f"Could not list folder: HTTP {r.status_code}")


--- data/processed/ ---
file   data/processed/readme.md (535 bytes)

--- data/raw/ ---
file   data/raw/P_ALB_CR.xpt (835,600 bytes)
file   data/raw/P_ALQ.xpt (719,360 bytes)
file   data/raw/P_BIOPRO.xpt (3,420,640 bytes)
file   data/raw/P_BMX.xpt (2,520,640 bytes)
file   data/raw/P_BPXO.xpt (1,039,840 bytes)
file   data/raw/P_CBC.xpt (2,427,760 bytes)
file   data/raw/P_DBQ.xpt (5,733,280 bytes)
file   data/raw/P_DEMO.xpt (3,614,720 bytes)
file   data/raw/P_DIQ.xpt (3,361,520 bytes)
file   data/raw/P_GHB.xpt (167,600 bytes)
file   data/raw/P_GLU.xpt (164,160 bytes)
file   data/raw/P_HDL.xpt (294,000 bytes)
file   data/raw/P_HIQ.xpt (1,745,440 bytes)
file   data/raw/P_HSCRP.xpt (331,760 bytes)
file   data/raw/P_HUQ.xpt (873,120 bytes)
file   data/raw/P_PAQ.xpt (1,321,440 bytes)
file   data/raw/P_RXQ_RX.xpt (16,154,000 bytes)
file   data/raw/P_SLQ.xpt (777,200 bytes)
file   data/raw/P_SMQ.xpt (1,428,560 bytes)
file   data/raw/P_TCHOL.xpt (294,000 bytes)
file   data/raw/P_TRIGLY.xpt (409,

In [6]:
# Cell 5: Inspect Notebook 1 for saved datasets and model results

notebook_url = (
    "https://raw.githubusercontent.com/"
    "MeghaKA/Trustworthy-Agentic-AI-for-Healthcare/"
    "main/notebooks/ckd/"
    "CKD_NB1_Data_Audit_Preparation_and_Baseline_Models.ipynb"
)

r = requests.get(notebook_url, timeout=30)
r.raise_for_status()
nb1 = r.json()

print("Notebook 1 cells:", len(nb1["cells"]))
print("\nSaved output files mentioned in the notebook:\n")

for i, cell in enumerate(nb1["cells"]):
    source = "".join(cell.get("source", []))
    matches = [
        line.strip()
        for line in source.splitlines()
        if any(term in line.lower() for term in [
            ".csv", ".parquet", ".joblib", ".pkl",
            "to_csv", "to_parquet", "dump("
        ])
    ]

    if matches:
        print(f"Cell {i + 1}:")
        for line in matches:
            print(" ", line[:200])

Notebook 1 cells: 40

Saved output files mentioned in the notebook:



In [7]:
# Cell 6: Find existing model evaluation outputs in Notebook 1

for i, cell in enumerate(nb1["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if any(term in source.lower() for term in [
        "cross_val", "roc_auc", "average_precision",
        "out-of-fold", "oof", "threshold", "classification_report"
    ]):
        print(f"\n{'=' * 60}")
        print(f"Cell {i + 1} — Code")
        print(source[:500])

        outputs = cell.get("outputs", [])
        for output in outputs:
            data = output.get("data", {})
            text_output = data.get("text/plain", [])
            if text_output:
                print("Output:")
                print("".join(text_output)[:2500])


Cell 33 — Code
# Cell 33: Check modelling-library availability

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,


Cell 37 — Code
# Cell 37: Evaluate the logistic regression baseline with 5-fold CV

from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv_results = cross_validate(
    logistic_pipeline,
    X_train,


In [8]:
# Cell 7: Read the existing raw-data documentation

readme_url = (
    "https://raw.githubusercontent.com/"
    "MeghaKA/Trustworthy-Agentic-AI-for-Healthcare/"
    "main/data/raw/readme.md"
)

r = requests.get(readme_url, timeout=30)
r.raise_for_status()

print(r.text)

# Raw NHANES Data

This directory contains the public-use NHANES source files used by Notebook 1 for data integration and preprocessing.

## Data Source

National Health and Nutrition Examination Survey (NHANES), 2017–March 2020 Pre-Pandemic cycle

Provided by the National Center for Health Statistics (NCHS), part of the U.S. Centers for Disease Control and Prevention (CDC).

Official source: https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx

## Included Files

The following source components are used by Notebook 1:

* P_ALB_CR.xpt — Urine Albumin and Creatinine
* P_ALQ.xpt — Alcohol Use
* P_BIOPRO.xpt — Standard Biochemistry Profile
* P_BMX.xpt — Body Measures
* P_BPXO.xpt — Blood Pressure
* P_CBC.xpt — Complete Blood Count
* P_DBQ.xpt — Diet Behavior and Nutrition
* P_DEMO.xpt — Demographics
* P_DIQ.xpt — Diabetes Questionnaire
* P_GHB.xpt — Glycohemoglobin
* P_GLU.xpt — Plasma Fasting Glucose
* P_HDL.xpt — HDL Cholesterol
* P_HIQ.xpt — Health Insurance
* P_HSCRP.xpt — High-Sensi

In [9]:
# Cell 8: Download the documented NHANES source files

import requests
from pathlib import Path

RAW_DIR = Path("/content/ckd_nb2_data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

base_url = (
    "https://raw.githubusercontent.com/"
    "MeghaKA/Trustworthy-Agentic-AI-for-Healthcare/"
    "main/data/raw/"
)

raw_files = [
    "P_ALB_CR.xpt", "P_ALQ.xpt", "P_BIOPRO.xpt",
    "P_BMX.xpt", "P_BPXO.xpt", "P_CBC.xpt",
    "P_DBQ.xpt", "P_DEMO.xpt", "P_DIQ.xpt",
    "P_GHB.xpt", "P_GLU.xpt", "P_HDL.xpt",
    "P_HIQ.xpt", "P_HSCRP.xpt", "P_HUQ.xpt",
    "P_PAQ.xpt", "P_RXQ_RX.xpt", "P_SLQ.xpt",
    "P_SMQ.xpt", "P_TCHOL.xpt", "P_TRIGLY.xpt",
]

for filename in raw_files:
    destination = RAW_DIR / filename
    response = requests.get(base_url + filename, timeout=60)
    response.raise_for_status()
    destination.write_bytes(response.content)
    print(f"Downloaded: {filename} ({destination.stat().st_size:,} bytes)")

print(f"\nTotal files: {len(raw_files)}")
print(f"Total size: {sum(p.stat().st_size for p in RAW_DIR.iterdir()):,} bytes")

Downloaded: P_ALB_CR.xpt (835,600 bytes)
Downloaded: P_ALQ.xpt (719,360 bytes)
Downloaded: P_BIOPRO.xpt (3,420,640 bytes)
Downloaded: P_BMX.xpt (2,520,640 bytes)
Downloaded: P_BPXO.xpt (1,039,840 bytes)
Downloaded: P_CBC.xpt (2,427,760 bytes)
Downloaded: P_DBQ.xpt (5,733,280 bytes)
Downloaded: P_DEMO.xpt (3,614,720 bytes)
Downloaded: P_DIQ.xpt (3,361,520 bytes)
Downloaded: P_GHB.xpt (167,600 bytes)
Downloaded: P_GLU.xpt (164,160 bytes)
Downloaded: P_HDL.xpt (294,000 bytes)
Downloaded: P_HIQ.xpt (1,745,440 bytes)
Downloaded: P_HSCRP.xpt (331,760 bytes)
Downloaded: P_HUQ.xpt (873,120 bytes)
Downloaded: P_PAQ.xpt (1,321,440 bytes)
Downloaded: P_RXQ_RX.xpt (16,154,000 bytes)
Downloaded: P_SLQ.xpt (777,200 bytes)
Downloaded: P_SMQ.xpt (1,428,560 bytes)
Downloaded: P_TCHOL.xpt (294,000 bytes)
Downloaded: P_TRIGLY.xpt (409,360 bytes)

Total files: 21
Total size: 47,634,000 bytes


In [10]:
# Cell 9: Locate the original dataset integration and target logic

for i, cell in enumerate(nb1["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    keywords = [
        "read_sas", "merge(", "SEQN", "eGFR",
        "URDACT", "URXUMA", "URXUCR",
        "ckd_research_indicator", "LBXSCR",
        "mean_systolic_bp", "predictors"
    ]

    if any(word.lower() in source.lower() for word in keywords):
        print(f"\n{'=' * 60}")
        print(f"Notebook 1 — Cell {i + 1}")
        print(source[:3000])


Notebook 1 — Cell 5
import pandas as pd
from pathlib import Path

RAW_DATA = PROJECT_ROOT / "data" / "raw"

# Kidney-related NHANES source files
kidney_files = [
    "P_ALB_CR.xpt",  # Urine albumin and creatinine
    "P_BIOPRO.xpt",  # Blood biochemistry, including kidney-related measures
    "P_DEMO.xpt",    # Demographic information and participant ID
]

for filename in kidney_files:
    file_path = RAW_DATA / filename

    print("\n" + "=" * 70)
    print("FILE:", filename)

    if not file_path.exists():
        print("File not found.")
        continue

    df = pd.read_sas(file_path, format="xport")

    print("Shape:", df.shape)
    print("\nAvailable columns:")
    print(df.columns.tolist())

    print("\nFirst 3 rows:")
    display(df.head(3))

    print("\nMissingness in each column (%):")
    missing = (df.isna().mean() * 100).round(2)
    display(missing.sort_values(ascending=False).to_frame("missing_percent"))

Notebook 1 — Cell 6
import numpy as np
import pandas as pd



In [11]:
# Cell 10: Inspect the saved train/test split results

cell_index = 34  # Notebook 1, Cell 35

cell = nb1["cells"][cell_index]

print("Code:")
print("".join(cell.get("source", [])))

print("\nSaved output:")
for output in cell.get("outputs", []):
    data = output.get("data", {})
    if "text" in output:
        print("".join(output["text"]))
    if "text/plain" in data:
        print("".join(data["text/plain"]))

Code:
# Cell 35: Prepare X and y, then verify target balance

from sklearn.model_selection import train_test_split

X = model_features[candidate_predictors].copy()
y = model_features[target_col].astype(int).copy()

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

print("\nOverall target distribution:")
print(y.value_counts().sort_index())

print("\nOverall target percentages:")
print((y.value_counts(normalize=True).sort_index() * 100).round(2))

# Reserve 20% for a held-out test set; stratify to preserve class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("\nTraining set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTest target distribution:")
print(y_test.value_counts().sort_index())

print("\nTraining target percentages:")
print((y_train.value_c

In [12]:
# Cell 11: Reconstruct and verify the original target cohort

import numpy as np
import pandas as pd

demo = pd.read_sas(
    RAW_DIR / "P_DEMO.xpt", format="xport"
)[["SEQN", "RIDAGEYR", "RIAGENDR"]]

biopro = pd.read_sas(
    RAW_DIR / "P_BIOPRO.xpt", format="xport"
)[["SEQN", "LBXSCR"]]

urine = pd.read_sas(
    RAW_DIR / "P_ALB_CR.xpt", format="xport"
)[["SEQN", "URDACT"]]

audit = (
    demo.merge(biopro, on="SEQN", how="left", validate="one_to_one")
        .merge(urine, on="SEQN", how="left", validate="one_to_one")
)

ckd_audit = audit.loc[audit["RIDAGEYR"] >= 18].copy()

scr = ckd_audit["LBXSCR"]
age = ckd_audit["RIDAGEYR"]
female = ckd_audit["RIAGENDR"].eq(2)
male = ckd_audit["RIAGENDR"].eq(1)

valid = scr.notna() & (scr > 0) & female.add(male).eq(1)

ckd_audit["eGFR_2021"] = np.nan

mask_f = valid & female
ckd_audit.loc[mask_f, "eGFR_2021"] = (
    142
    * np.minimum(scr[mask_f] / 0.7, 1) ** (-0.241)
    * np.maximum(scr[mask_f] / 0.7, 1) ** (-1.200)
    * 0.9938 ** age[mask_f]
    * 1.012
)

mask_m = valid & male
ckd_audit.loc[mask_m, "eGFR_2021"] = (
    142
    * np.minimum(scr[mask_m] / 0.9, 1) ** (-0.302)
    * np.maximum(scr[mask_m] / 0.9, 1) ** (-1.200)
    * 0.9938 ** age[mask_m]
)

low_egfr = ckd_audit["eGFR_2021"] < 60
high_uacr = ckd_audit["URDACT"] >= 30
both_observed = (
    ckd_audit["eGFR_2021"].notna()
    & ckd_audit["URDACT"].notna()
)

positive = low_egfr.eq(True) | high_uacr.eq(True)

ckd_audit["ckd_research_indicator"] = np.nan
ckd_audit.loc[positive, "ckd_research_indicator"] = 1
ckd_audit.loc[both_observed & ~positive, "ckd_research_indicator"] = 0

print("Adult cohort:", len(ckd_audit))
print("Unique participant IDs:", ckd_audit["SEQN"].nunique())
print("Labelled participants:", ckd_audit["ckd_research_indicator"].notna().sum())
print("Unlabelled participants:", ckd_audit["ckd_research_indicator"].isna().sum())
print("\nLabel counts:")
print(ckd_audit["ckd_research_indicator"].value_counts(dropna=False).sort_index())

assert len(ckd_audit) == 9693
assert ckd_audit["SEQN"].nunique() == 9693
assert int(ckd_audit["ckd_research_indicator"].notna().sum()) == 8267
assert int((ckd_audit["ckd_research_indicator"] == 0).sum()) == 6644
assert int((ckd_audit["ckd_research_indicator"] == 1).sum()) == 1623

print("\nAll cohort and target-count checks passed.")

Adult cohort: 9693
Unique participant IDs: 9693
Labelled participants: 8267
Unlabelled participants: 1426

Label counts:
ckd_research_indicator
0.0    6644
1.0    1623
NaN    1426
Name: count, dtype: int64

All cohort and target-count checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [13]:
# Cell 12: Load predictor source tables and merge by participant ID

source_columns = {
    "P_BMX.xpt": ["SEQN", "BMXBMI", "BMXWAIST"],
    "P_BPXO.xpt": [
        "SEQN", "BPXOSY1", "BPXOSY2", "BPXOSY3",
        "BPXODI1", "BPXODI2", "BPXODI3"
    ],
    "P_DIQ.xpt": ["SEQN", "DIQ010"],
    "P_HIQ.xpt": ["SEQN", "HIQ011"],
    "P_HUQ.xpt": ["SEQN", "HUQ010"],
    "P_PAQ.xpt": [
        "SEQN", "PAQ605", "PAQ620", "PAQ650", "PAQ665"
    ],
    "P_SMQ.xpt": ["SEQN", "SMQ020"],
    "P_GHB.xpt": ["SEQN", "LBXGH"],
}

predictors = pd.DataFrame({"SEQN": ckd_audit["SEQN"]})

for filename, columns in source_columns.items():
    table = pd.read_sas(RAW_DIR / filename, format="xport")
    table = table[columns].copy()

    if table["SEQN"].duplicated().any():
        raise ValueError(f"Duplicate participant IDs found in {filename}")

    predictors = predictors.merge(
        table,
        on="SEQN",
        how="left",
        validate="one_to_one"
    )

print("Adult cohort rows:", len(ckd_audit))
print("Merged predictor rows:", len(predictors))
print("Unique participant IDs:", predictors["SEQN"].nunique())
print("Duplicate participant IDs:", predictors["SEQN"].duplicated().sum())
print("\nPredictor columns loaded:")
print(predictors.columns.tolist())

assert len(predictors) == len(ckd_audit)
assert predictors["SEQN"].nunique() == len(ckd_audit)

print("\nPredictor merge checks passed.")

Adult cohort rows: 9693
Merged predictor rows: 9693
Unique participant IDs: 9693
Duplicate participant IDs: 0

Predictor columns loaded:
['SEQN', 'BMXBMI', 'BMXWAIST', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3', 'BPXODI1', 'BPXODI2', 'BPXODI3', 'DIQ010', 'HIQ011', 'HUQ010', 'PAQ605', 'PAQ620', 'PAQ650', 'PAQ665', 'SMQ020', 'LBXGH']

Predictor merge checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [14]:
# Cell 13: Derive mean blood-pressure features

systolic_cols = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]
diastolic_cols = ["BPXODI1", "BPXODI2", "BPXODI3"]

predictors["mean_systolic_bp"] = predictors[systolic_cols].mean(
    axis=1, skipna=True
)
predictors["mean_diastolic_bp"] = predictors[diastolic_cols].mean(
    axis=1, skipna=True
)

print("Derived feature checks:")
print(
    predictors[
        ["mean_systolic_bp", "mean_diastolic_bp"]
    ].describe().round(2)
)

print("\nMissing values in derived features:")
print(
    predictors[
        ["mean_systolic_bp", "mean_diastolic_bp"]
    ].isna().sum()
)

print("\nParticipants with at least one systolic reading:",
      predictors[systolic_cols].notna().any(axis=1).sum())

print("Participants with at least one diastolic reading:",
      predictors[diastolic_cols].notna().any(axis=1).sum())

Derived feature checks:
       mean_systolic_bp  mean_diastolic_bp
count           8024.00            8024.00
mean             124.22              74.35
std               19.32              11.62
min               76.33              41.33
25%              110.33              66.33
50%              121.33              73.67
75%              134.67              81.33
max              218.67             143.67

Missing values in derived features:
mean_systolic_bp     1669
mean_diastolic_bp    1669
dtype: int64

Participants with at least one systolic reading: 8024
Participants with at least one diastolic reading: 8024


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [15]:
# Cell 14 : Assemble the modelling dataset

candidate_predictors = [
    "RIDAGEYR", "RIAGENDR", "BMXBMI", "BMXWAIST",
    "mean_systolic_bp", "mean_diastolic_bp",
    "DIQ010", "HIQ011", "HUQ010",
    "PAQ605", "PAQ620", "PAQ650", "PAQ665",
    "SMQ020", "LBXGH",
]

target_col = "ckd_research_indicator"

# Start with the original demographic and target columns.
model_data = ckd_audit[
    ["SEQN", "RIDAGEYR", "RIAGENDR", target_col]
].merge(
    predictors.drop(
        columns=["RIDAGEYR", "RIAGENDR"],
        errors="ignore"
    ),
    on="SEQN",
    how="left",
    validate="one_to_one"
)

model_data = model_data.loc[
    model_data[target_col].notna()
].copy()

X = model_data[candidate_predictors].copy()
y = model_data[target_col].astype(int).copy()

print("Model dataset shape:", model_data.shape)
print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("\nTarget counts:")
print(y.value_counts().sort_index())
print("\nDuplicate participant IDs:", model_data["SEQN"].duplicated().sum())

assert X.shape == (8267, 15)
assert y.value_counts().to_dict() == {0: 6644, 1: 1623}
assert model_data["SEQN"].is_unique

print("\nAll modelling dataset checks passed.")

Model dataset shape: (8267, 23)
Feature matrix shape: (8267, 15)
Target vector shape: (8267,)

Target counts:
ckd_research_indicator
0    6644
1    1623
Name: count, dtype: int64

Duplicate participant IDs: 0

All modelling dataset checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [16]:
# Cell 15: Reproduce the original train/test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining target counts:")
print(y_train.value_counts().sort_index())

print("\nTest target counts:")
print(y_test.value_counts().sort_index())

print("\nTrain/test index overlap:",
      len(set(X_train.index) & set(X_test.index)))

assert X_train.shape == (6613, 15)
assert X_test.shape == (1654, 15)
assert y_train.value_counts().to_dict() == {0: 5315, 1: 1298}
assert y_test.value_counts().to_dict() == {0: 1329, 1: 325}
assert len(set(X_train.index) & set(X_test.index)) == 0

print("\nOriginal split checks passed.")

Training shape: (6613, 15)
Test shape: (1654, 15)

Training target counts:
ckd_research_indicator
0    5315
1    1298
Name: count, dtype: int64

Test target counts:
ckd_research_indicator
0    1329
1     325
Name: count, dtype: int64

Train/test index overlap: 0

Original split checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:
# Cell 16: Define leakage-safe preprocessing

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = [
    "RIDAGEYR",
    "BMXBMI",
    "BMXWAIST",
    "mean_systolic_bp",
    "mean_diastolic_bp",
    "LBXGH",
]

categorical_features = [
    "RIAGENDR",
    "DIQ010",
    "HIQ011",
    "HUQ010",
    "PAQ605",
    "PAQ620",
    "PAQ650",
    "PAQ665",
    "SMQ020",
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total original predictors:",
      len(numeric_features) + len(categorical_features))
print("Preprocessor defined:", isinstance(preprocessor, ColumnTransformer))

assert len(numeric_features) == 6
assert len(categorical_features) == 9
assert set(numeric_features + categorical_features) == set(X_train.columns)

print("\nPreprocessing definition checks passed.")

Numeric features: 6
Categorical features: 9
Total original predictors: 15
Preprocessor defined: True

Preprocessing definition checks passed.


In [18]:
# Cell 17: Audit training predictor missingness and categorical values

print("Missing values in training predictors:")
missing = X_train.isna().sum()
missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (missing / len(X_train) * 100).round(2),
})
print(missing_report.sort_values("missing_percent", ascending=False))

print("\nCategorical values observed in training data:")
for col in categorical_features:
    values = sorted(X_train[col].dropna().unique().tolist())
    print(f"{col}: {values}")

print("\nTraining target prevalence:",
      round(y_train.mean() * 100, 2), "%")

Missing values in training predictors:
                   missing_count  missing_percent
mean_systolic_bp             603             9.12
mean_diastolic_bp            603             9.12
BMXWAIST                     293             4.43
BMXBMI                        94             1.42
LBXGH                         68             1.03
RIAGENDR                       0             0.00
RIDAGEYR                       0             0.00
DIQ010                         0             0.00
HIQ011                         0             0.00
PAQ605                         0             0.00
HUQ010                         0             0.00
PAQ620                         0             0.00
PAQ650                         0             0.00
PAQ665                         0             0.00
SMQ020                         0             0.00

Categorical values observed in training data:
RIAGENDR: [1.0, 2.0]
DIQ010: [1.0, 2.0, 3.0, 9.0]
HIQ011: [1.0, 2.0, 7.0, 9.0]
HUQ010: [1.0, 2.0, 3.0, 4.0, 5.0, 7

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [19]:
# Cell 18: Apply the original categorical special-code rules

special_code_columns = [
    "DIQ010",
    "HIQ011",
    "HUQ010",
    "PAQ605",
    "PAQ620",
    "PAQ650",
    "PAQ665",
    "SMQ020",
]

# Count special codes before recoding.
print("Special codes found before recoding:")
for col in special_code_columns:
    count = X_train[col].isin([7, 9]).sum()
    if count:
        print(f"{col}: {count}")

# Recode on copies of both partitions; do not modify the original model_data.
X_train = X_train.copy()
X_test = X_test.copy()

for col in special_code_columns:
    X_train[col] = X_train[col].replace([7, 9], np.nan)
    X_test[col] = X_test[col].replace([7, 9], np.nan)

print("\nRemaining special codes in training data:")
remaining = {
    col: int(X_train[col].isin([7, 9]).sum())
    for col in special_code_columns
}
print(remaining)

print("\nMissing values after recoding (training):")
print(X_train[special_code_columns].isna().sum())

assert all(count == 0 for count in remaining.values())
assert len(X_train) == 6613
assert len(X_test) == 1654

print("\nSpecial-code recoding checks passed.")

Special codes found before recoding:
DIQ010: 4
HIQ011: 17
HUQ010: 7
PAQ605: 5
PAQ620: 7
PAQ665: 2
SMQ020: 2

Remaining special codes in training data:
{'DIQ010': 0, 'HIQ011': 0, 'HUQ010': 0, 'PAQ605': 0, 'PAQ620': 0, 'PAQ650': 0, 'PAQ665': 0, 'SMQ020': 0}

Missing values after recoding (training):
DIQ010     4
HIQ011    17
HUQ010     7
PAQ605     5
PAQ620     7
PAQ650     0
PAQ665     2
SMQ020     2
dtype: int64

Special-code recoding checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [20]:
# Cell 19: Verify recoding and target integrity

print("Remaining special codes in test data:")
test_remaining = {
    col: int(X_test[col].isin([7, 9]).sum())
    for col in special_code_columns
}
print(test_remaining)

print("\nMissing categorical values in test data:")
print(X_test[special_code_columns].isna().sum())

print("\nTraining rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training positive cases:", int(y_train.sum()))
print("Test positive cases:", int(y_test.sum()))

assert all(count == 0 for count in test_remaining.values())
assert len(X_train) == 6613 and len(X_test) == 1654
assert int(y_train.sum()) == 1298
assert int(y_test.sum()) == 325

print("\nTest-set and target-integrity checks passed.")

Remaining special codes in test data:
{'DIQ010': 0, 'HIQ011': 0, 'HUQ010': 0, 'PAQ605': 0, 'PAQ620': 0, 'PAQ650': 0, 'PAQ665': 0, 'SMQ020': 0}

Missing categorical values in test data:
DIQ010    0
HIQ011    2
HUQ010    2
PAQ605    1
PAQ620    0
PAQ650    0
PAQ665    0
SMQ020    2
dtype: int64

Training rows: 6613
Test rows: 1654
Training positive cases: 1298
Test positive cases: 325

Test-set and target-integrity checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [21]:
# Cell 20: Five-fold cross-validation for baseline logistic regression

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    )),
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv_results = cross_validate(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    error_score="raise",
)

summary = pd.DataFrame({
    metric: [
        cv_results[f"test_{metric}"].mean(),
        cv_results[f"test_{metric}"].std(),
    ]
    for metric in scoring
}, index=["Mean", "Std"]).T

print("Baseline logistic regression — 5-fold CV")
print("(Training data only; mean and standard deviation)")
print(summary.round(4))

Baseline logistic regression — 5-fold CV
(Training data only; mean and standard deviation)
                     Mean     Std
accuracy           0.8216  0.0088
precision          0.6305  0.0724
recall             0.2296  0.0081
f1                 0.3359  0.0170
roc_auc            0.7668  0.0189
average_precision  0.4910  0.0288


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [22]:
# Cell 21: Class-weighted logistic regression — 5-fold CV

weighted_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )),
])

weighted_results = cross_validate(
    weighted_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    error_score="raise",
)

weighted_summary = pd.DataFrame({
    metric: [
        weighted_results[f"test_{metric}"].mean(),
        weighted_results[f"test_{metric}"].std(),
    ]
    for metric in scoring
}, index=["Mean", "Std"]).T

print("Class-weighted logistic regression — 5-fold CV")
print(weighted_summary.round(4))

Class-weighted logistic regression — 5-fold CV
                     Mean     Std
accuracy           0.7085  0.0181
precision          0.3727  0.0202
recall             0.7064  0.0334
f1                 0.4877  0.0230
roc_auc            0.7659  0.0189
average_precision  0.4864  0.0279


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [23]:
# Cell 22: Out-of-fold probabilities for threshold analysis

from sklearn.model_selection import cross_val_predict

oof_probabilities = cross_val_predict(
    weighted_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

print("OOF probability count:", len(oof_probabilities))
print("Training target count:", len(y_train))
print("Probability range:",
      round(float(oof_probabilities.min()), 4),
      "to",
      round(float(oof_probabilities.max()), 4))
print("Missing OOF probabilities:",
      int(np.isnan(oof_probabilities).sum()))

assert len(oof_probabilities) == len(y_train)
assert np.isfinite(oof_probabilities).all()

print("\nOOF probability checks passed.")

OOF probability count: 6613
Training target count: 6613
Probability range: 0.0833 to 0.9721
Missing OOF probabilities: 0

OOF probability checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [24]:
# Cell 23: Out-of-fold threshold comparison

from sklearn.metrics import precision_score, recall_score, f1_score

threshold_rows = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60]:
    predictions = (oof_probabilities >= threshold).astype(int)

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(
            y_train, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_train, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_train, predictions, zero_division=0
        ),
        "predicted_positive": int(predictions.sum()),
    })

threshold_results = pd.DataFrame(threshold_rows)

print("Out-of-fold threshold results (training data only):")
print(threshold_results.round(4).to_string(index=False))

Out-of-fold threshold results (training data only):
 threshold  precision  recall     f1  predicted_positive
       0.2     0.2245  0.9407 0.3625                5438
       0.3     0.2674  0.8767 0.4098                4256
       0.4     0.3141  0.7989 0.4510                3301
       0.5     0.3722  0.7065 0.4875                2464
       0.6     0.4398  0.5794 0.5000                1710


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [25]:
# Cell 24: Inspect the original Notebook 1 threshold-analysis code

for i in [39]:
    cell = nb1["cells"][i]
    print(f"Notebook 1 cell {i + 1} source:\n")
    print("".join(cell.get("source", [])))

Notebook 1 cell 40 source:

# Cell 40: Obtain out-of-fold probabilities for threshold analysis

from sklearn.base import clone
from sklearn.model_selection import cross_val_predict

oof_probabilities = cross_val_predict(
    clone(logistic_pipeline),
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)

oof_positive_probabilities = oof_probabilities[:, 1]

oof_threshold_rows = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60]:
    oof_predictions = (
        oof_positive_probabilities >= threshold
    ).astype(int)

    oof_threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(
            y_train, oof_predictions, zero_division=0
        ),
        "recall": recall_score(
            y_train, oof_predictions, zero_division=0
        ),
        "f1": f1_score(
            y_train, oof_predictions, zero_division=0
        ),
        "predicted_positive_count": int(oof_predictions.sum()),
    })

oof_threshold_table = 

In [26]:
# Cell 25: Reproduce Notebook 1's OOF threshold analysis

from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

oof_probabilities_original = cross_val_predict(
    clone(baseline_model),
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

oof_threshold_rows_original = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60]:
    oof_predictions = (
        oof_probabilities_original >= threshold
    ).astype(int)

    oof_threshold_rows_original.append({
        "threshold": threshold,
        "precision": precision_score(
            y_train, oof_predictions, zero_division=0
        ),
        "recall": recall_score(
            y_train, oof_predictions, zero_division=0
        ),
        "f1": f1_score(
            y_train, oof_predictions, zero_division=0
        ),
        "predicted_positive_count": int(oof_predictions.sum()),
    })

oof_threshold_table_original = pd.DataFrame(
    oof_threshold_rows_original
)

print("Original-model OOF threshold analysis:")
print(oof_threshold_table_original.round(4).to_string(index=False))

print("\nOOF probabilities generated:", len(oof_probabilities_original))
print("Training observations:", len(y_train))
print("Held-out test set used:", False)

Original-model OOF threshold analysis:
 threshold  precision  recall     f1  predicted_positive_count
       0.2     0.3730  0.7057 0.4880                      2456
       0.3     0.4658  0.5401 0.5002                      1505
       0.4     0.5341  0.3559 0.4272                       865
       0.5     0.6234  0.2296 0.3356                       478
       0.6     0.7143  0.1194 0.2046                       217

OOF probabilities generated: 6613
Training observations: 6613
Held-out test set used: False


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [27]:
# Cell 26: Compare standard and class-weighted models at threshold 0.50

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

model_comparison_rows = []

for model_name, probabilities in [
    ("Standard Logistic Regression", oof_probabilities_original),
    ("Class-weighted Logistic Regression", oof_probabilities),
]:
    predictions = (probabilities >= 0.50).astype(int)

    model_comparison_rows.append({
        "model": model_name,
        "accuracy": accuracy_score(y_train, predictions),
        "precision": precision_score(
            y_train, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_train, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_train, predictions, zero_division=0
        ),
        "predicted_positives": int(predictions.sum()),
    })

model_comparison = pd.DataFrame(model_comparison_rows)

print("OOF model comparison at threshold 0.50:")
print(model_comparison.round(4).to_string(index=False))

OOF model comparison at threshold 0.50:
                             model  accuracy  precision  recall     f1  predicted_positives
      Standard Logistic Regression    0.8216     0.6234  0.2296 0.3356                  478
Class-weighted Logistic Regression    0.7085     0.3722  0.7065 0.4875                 2464


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [28]:
# Cell 27: Final pre-test integrity checks

from sklearn.utils.validation import check_is_fitted

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training positive cases:", int(y_train.sum()))
print("Test positive cases:", int(y_test.sum()))

print("Training/test participant overlap:",
      len(set(model_data.loc[X_train.index, "SEQN"]) &
          set(model_data.loc[X_test.index, "SEQN"])))

print("Standard pipeline fitted outside CV:", end=" ")
try:
    check_is_fitted(baseline_model.named_steps["classifier"])
    print(True)
except Exception:
    print(False)

assert len(X_train) == 6613
assert len(X_test) == 1654
assert int(y_train.sum()) == 1298
assert int(y_test.sum()) == 325
assert len(set(model_data.loc[X_train.index, "SEQN"]) &
           set(model_data.loc[X_test.index, "SEQN"])) == 0

print("\nPre-test integrity checks passed.")

Training rows: 6613
Test rows: 1654
Training positive cases: 1298
Test positive cases: 325
Training/test participant overlap: 0
Standard pipeline fitted outside CV: False

Pre-test integrity checks passed.


In [29]:
# Cell 28: Fit both logistic regression models on training data only

baseline_model.fit(X_train, y_train)
weighted_model.fit(X_train, y_train)

print("Standard Logistic Regression: fitted")
print("Class-weighted Logistic Regression: fitted")

print("\nTraining rows used:", len(X_train))
print("Held-out test rows used for fitting: 0")

assert len(X_train) == 6613
assert len(X_test) == 1654

print("\nModel-fitting checks passed.")

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Standard Logistic Regression: fitted
Class-weighted Logistic Regression: fitted

Training rows used: 6613
Held-out test rows used for fitting: 0

Model-fitting checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [30]:
# Cell 29: Held-out test evaluation at threshold 0.50

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

test_results = []
test_probabilities = {}

for model_name, model in [
    ("Standard Logistic Regression", baseline_model),
    ("Class-weighted Logistic Regression", weighted_model),
]:
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.50).astype(int)

    test_probabilities[model_name] = probabilities

    test_results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_test, probabilities),
        "average_precision": average_precision_score(
            y_test, probabilities
        ),
        "predicted_positives": int(predictions.sum()),
    })

test_results_df = pd.DataFrame(test_results)

print("Held-out test results at threshold 0.50:")
print(test_results_df.round(4).to_string(index=False))

print("\nTest participants:", len(y_test))
print("Actual positive cases:", int(y_test.sum()))

Held-out test results at threshold 0.50:
                             model  accuracy  precision  recall     f1  roc_auc  average_precision  predicted_positives
      Standard Logistic Regression    0.8174     0.6162  0.1877 0.2877   0.7994             0.5084                   99
Class-weighted Logistic Regression    0.7152     0.3866  0.7662 0.5139   0.7994             0.5041                  644

Test participants: 1654
Actual positive cases: 325


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [31]:
# Cell 30: Test-set threshold sensitivity for the class-weighted model

from sklearn.metrics import precision_score, recall_score, f1_score

weighted_test_probabilities = test_probabilities[
    "Class-weighted Logistic Regression"
]

test_threshold_rows = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60]:
    predictions = (
        weighted_test_probabilities >= threshold
    ).astype(int)

    test_threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "predicted_positives": int(predictions.sum()),
    })

test_threshold_results = pd.DataFrame(test_threshold_rows)

print("Class-weighted model: test threshold sensitivity")
print(test_threshold_results.round(4).to_string(index=False))

Class-weighted model: test threshold sensitivity
 threshold  precision  recall     f1  predicted_positives
       0.2     0.2235  0.9538 0.3621                 1387
       0.3     0.2835  0.9169 0.4331                 1051
       0.4     0.3333  0.8646 0.4812                  843
       0.5     0.3866  0.7662 0.5139                  644
       0.6     0.4665  0.6215 0.5330                  433


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [32]:
# Cell 31: Confusion matrices on the held-out test set at threshold 0.50

from sklearn.metrics import confusion_matrix

for model_name, model in [
    ("Standard Logistic Regression", baseline_model),
    ("Class-weighted Logistic Regression", weighted_model),
]:
    probabilities = test_probabilities[model_name]
    predictions = (probabilities >= 0.50).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()

    print(f"\n{model_name}")
    print(f"True negatives (TN):  {tn}")
    print(f"False positives (FP): {fp}")
    print(f"False negatives (FN): {fn}")
    print(f"True positives (TP):  {tp}")
    print(f"Total test participants: {tn + fp + fn + tp}")

    assert tn + fp + fn + tp == len(y_test)

print("\nConfusion-matrix checks passed.")


Standard Logistic Regression
True negatives (TN):  1291
False positives (FP): 38
False negatives (FN): 264
True positives (TP):  61
Total test participants: 1654

Class-weighted Logistic Regression
True negatives (TN):  934
False positives (FP): 395
False negatives (FN): 76
True positives (TP):  249
Total test participants: 1654

Confusion-matrix checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [33]:
# Cell 32: Additional test-set metrics at threshold 0.50

from sklearn.metrics import confusion_matrix

additional_metrics = []

for model_name in [
    "Standard Logistic Regression",
    "Class-weighted Logistic Regression",
]:
    probabilities = test_probabilities[model_name]
    predictions = (probabilities >= 0.50).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()

    additional_metrics.append({
        "model": model_name,
        "specificity": tn / (tn + fp) if (tn + fp) else float("nan"),
        "negative_predictive_value": (
            tn / (tn + fn) if (tn + fn) else float("nan")
        ),
        "false_positive_rate": (
            fp / (fp + tn) if (fp + tn) else float("nan")
        ),
    })

additional_metrics_df = pd.DataFrame(additional_metrics)

print("Additional held-out test metrics at threshold 0.50:")
print(additional_metrics_df.round(4).to_string(index=False))

Additional held-out test metrics at threshold 0.50:
                             model  specificity  negative_predictive_value  false_positive_rate
      Standard Logistic Regression       0.9714                     0.8302               0.0286
Class-weighted Logistic Regression       0.7028                     0.9248               0.2972


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [34]:
# Cell 33: Probability quality on the held-out test set

from sklearn.metrics import brier_score_loss, log_loss

probability_quality_rows = []

for model_name, probabilities in test_probabilities.items():
    probability_quality_rows.append({
        "model": model_name,
        "brier_score": brier_score_loss(y_test, probabilities),
        "log_loss": log_loss(
            y_test, probabilities, labels=[0, 1]
        ),
    })

probability_quality_df = pd.DataFrame(probability_quality_rows)

print("Held-out test probability-quality metrics:")
print(probability_quality_df.round(4).to_string(index=False))

Held-out test probability-quality metrics:
                             model  brier_score  log_loss
      Standard Logistic Regression       0.1259    0.4009
Class-weighted Logistic Regression       0.1867    0.5567


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [35]:
# Cell 34: Calibration-bin comparison on the held-out test set

from sklearn.calibration import calibration_curve

calibration_rows = []

for model_name, probabilities in test_probabilities.items():
    observed_fraction, mean_predicted = calibration_curve(
        y_test,
        probabilities,
        n_bins=5,
        strategy="quantile",
    )

    for bin_number, (predicted, observed) in enumerate(
        zip(mean_predicted, observed_fraction), start=1
    ):
        calibration_rows.append({
            "model": model_name,
            "bin": bin_number,
            "mean_predicted_probability": predicted,
            "observed_positive_fraction": observed,
        })

calibration_df = pd.DataFrame(calibration_rows)

print("Calibration-bin results (held-out test set):")
print(calibration_df.round(4).to_string(index=False))

Calibration-bin results (held-out test set):


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


                             model  bin  mean_predicted_probability  observed_positive_fraction
      Standard Logistic Regression    1                      0.0423                      0.0483
      Standard Logistic Regression    2                      0.0782                      0.0483
      Standard Logistic Regression    3                      0.1455                      0.1242
      Standard Logistic Regression    4                      0.2525                      0.2477
      Standard Logistic Regression    5                      0.4608                      0.5136
Class-weighted Logistic Regression    1                      0.1653                      0.0514
Class-weighted Logistic Regression    2                      0.2665                      0.0393
Class-weighted Logistic Regression    3                      0.4085                      0.1242
Class-weighted Logistic Regression    4                      0.5684                      0.2598
Class-weighted Logistic Regression    5 

In [36]:
# Cell 35: Consolidated held-out test evaluation

final_metric_rows = []

for model_name, model in [
    ("Standard Logistic Regression", baseline_model),
    ("Class-weighted Logistic Regression", weighted_model),
]:
    probabilities = test_probabilities[model_name]
    predictions = (probabilities >= 0.50).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()

    final_metric_rows.append({
        "model": model_name,
        "threshold": 0.50,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "specificity": tn / (tn + fp),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
        "average_precision": average_precision_score(
            y_test, probabilities
        ),
        "brier_score": brier_score_loss(y_test, probabilities),
        "log_loss": log_loss(y_test, probabilities, labels=[0, 1]),
    })

final_test_summary = pd.DataFrame(final_metric_rows)

print("Consolidated held-out test results:")
print(final_test_summary.round(4).to_string(index=False))

assert len(final_test_summary) == 2
assert final_test_summary["model"].nunique() == 2

print("\nConsolidated evaluation checks passed.")

Consolidated held-out test results:
                             model  threshold  accuracy  precision  recall  specificity     f1  roc_auc  average_precision  brier_score  log_loss
      Standard Logistic Regression        0.5    0.8174     0.6162  0.1877       0.9714 0.2877   0.7994             0.5084       0.1259    0.4009
Class-weighted Logistic Regression        0.5    0.7152     0.3866  0.7662       0.7028 0.5139   0.7994             0.5041       0.1867    0.5567

Consolidated evaluation checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [37]:
# Cell 36: Inspect class-weighted Logistic Regression coefficients

fitted_preprocessor = weighted_model.named_steps["preprocessor"]
fitted_classifier = weighted_model.named_steps["classifier"]

transformed_feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = fitted_classifier.coef_[0]

coefficient_df = pd.DataFrame({
    "feature": transformed_feature_names,
    "coefficient": coefficients,
})

coefficient_df["absolute_coefficient"] = (
    coefficient_df["coefficient"].abs()
)

coefficient_df = coefficient_df.sort_values(
    "absolute_coefficient", ascending=False
).reset_index(drop=True)

print("Top 15 coefficients by absolute magnitude:")
print(
    coefficient_df[
        ["feature", "coefficient", "absolute_coefficient"]
    ].head(15).round(4).to_string(index=False)
)

print("\nLargest positive coefficients:")
print(
    coefficient_df.nlargest(10, "coefficient")[
        ["feature", "coefficient"]
    ].round(4).to_string(index=False)
)

print("\nLargest negative coefficients:")
print(
    coefficient_df.nsmallest(10, "coefficient")[
        ["feature", "coefficient"]
    ].round(4).to_string(index=False)
)

assert len(transformed_feature_names) == len(coefficients)
print("\nCoefficient extraction checks passed.")

Top 15 coefficients by absolute magnitude:
                  feature  coefficient  absolute_coefficient
        numeric__RIDAGEYR       0.6065                0.6065
  categorical__HUQ010_5.0       0.5373                0.5373
  categorical__DIQ010_1.0       0.4244                0.4244
  categorical__HUQ010_2.0      -0.3835                0.3835
  categorical__HUQ010_1.0      -0.3309                0.3309
  categorical__DIQ010_3.0      -0.3155                0.3155
numeric__mean_systolic_bp       0.2712                0.2712
           numeric__LBXGH       0.2477                0.2477
          numeric__BMXBMI       0.1551                0.1551
  categorical__DIQ010_2.0      -0.1521                0.1521
        numeric__BMXWAIST      -0.1495                0.1495
  categorical__PAQ605_1.0      -0.1292                0.1292
  categorical__HUQ010_4.0       0.0953                0.0953
categorical__RIAGENDR_1.0      -0.0877                0.0877
  categorical__PAQ605_2.0       0.0860    

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [38]:
# Cell 37: Feature and leakage integrity checks

forbidden_predictors = {
    "LBXSCR",
    "eGFR_2021",
    "URDACT",
    "URXUMA",
    "URXUCR",
    "low_eGFR_marker",
    "high_UACR_marker",
    "ckd_research_indicator",
}

assert len(transformed_feature_names) == len(coefficients)
assert np.isfinite(coefficients).all()
assert not (set(candidate_predictors) & forbidden_predictors)

print("Candidate predictors:", len(candidate_predictors))
print("Transformed model features:", len(transformed_feature_names))
print("Coefficient count:", len(coefficients))
print("Non-finite coefficients:", int((~np.isfinite(coefficients)).sum()))
print(
    "Forbidden target/leakage predictors included:",
    sorted(set(candidate_predictors) & forbidden_predictors),
)
print("\nFeature integrity checks passed.")

Candidate predictors: 15
Transformed model features: 28
Coefficient count: 28
Non-finite coefficients: 0
Forbidden target/leakage predictors included: []

Feature integrity checks passed.


In [39]:
# Cell 38: Correlation check for numeric predictors (training set only)

numeric_correlation = X_train[numeric_features].corr()

print("Correlation matrix for numeric predictors:")
print(numeric_correlation.round(3).to_string())

# Report predictor pairs with absolute correlation >= 0.70
correlation_pairs = []

for i, feature_a in enumerate(numeric_features):
    for feature_b in numeric_features[i + 1:]:
        correlation = numeric_correlation.loc[feature_a, feature_b]

        if pd.notna(correlation) and abs(correlation) >= 0.70:
            correlation_pairs.append({
                "feature_1": feature_a,
                "feature_2": feature_b,
                "correlation": correlation,
            })

correlation_pairs_df = pd.DataFrame(
    correlation_pairs,
    columns=["feature_1", "feature_2", "correlation"],
)

print("\nNumeric predictor pairs with |correlation| >= 0.70:")
if correlation_pairs_df.empty:
    print("None detected at this threshold.")
else:
    print(correlation_pairs_df.round(3).to_string(index=False))

print("\nCorrelation check completed.")

Correlation matrix for numeric predictors:
                   RIDAGEYR  BMXBMI  BMXWAIST  mean_systolic_bp  mean_diastolic_bp  LBXGH
RIDAGEYR              1.000   0.031     0.199             0.456              0.050  0.299
BMXBMI                0.031   1.000     0.906             0.036              0.222  0.171
BMXWAIST              0.199   0.906     1.000             0.122              0.244  0.242
mean_systolic_bp      0.456   0.036     0.122             1.000              0.612  0.174
mean_diastolic_bp     0.050   0.222     0.244             0.612              1.000  0.059
LBXGH                 0.299   0.171     0.242             0.174              0.059  1.000

Numeric predictor pairs with |correlation| >= 0.70:
feature_1 feature_2  correlation
   BMXBMI  BMXWAIST        0.906

Correlation check completed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [40]:
# Cell 39: Bootstrap coefficient stability on the training set

from sklearn.base import clone

BOOTSTRAP_REPEATS = 30
bootstrap_rng = np.random.default_rng(SEED)

bootstrap_coefficients = []

for bootstrap_id in range(BOOTSTRAP_REPEATS):
    sampled_positions = bootstrap_rng.choice(
        len(X_train),
        size=len(X_train),
        replace=True,
    )

    X_boot = X_train.iloc[sampled_positions]
    y_boot = y_train.iloc[sampled_positions]

    # Skip a resample if it contains only one target class.
    if y_boot.nunique() < 2:
        continue

    bootstrap_model = clone(weighted_model)
    bootstrap_model.fit(X_boot, y_boot)

    bootstrap_coefficients.append(
        bootstrap_model.named_steps["classifier"].coef_[0]
    )

bootstrap_coefficients = np.asarray(bootstrap_coefficients)

assert bootstrap_coefficients.ndim == 2
assert bootstrap_coefficients.shape[1] == len(transformed_feature_names)

bootstrap_summary = pd.DataFrame({
    "feature": transformed_feature_names,
    "original_coefficient": coefficients,
    "bootstrap_mean": bootstrap_coefficients.mean(axis=0),
    "bootstrap_std": bootstrap_coefficients.std(axis=0, ddof=1),
    "bootstrap_q025": np.percentile(bootstrap_coefficients, 2.5, axis=0),
    "bootstrap_q975": np.percentile(bootstrap_coefficients, 97.5, axis=0),
})

bootstrap_summary["sign_stability"] = np.maximum(
    (bootstrap_coefficients > 0).mean(axis=0),
    (bootstrap_coefficients < 0).mean(axis=0),
)

bootstrap_summary["original_abs_coefficient"] = (
    bootstrap_summary["original_coefficient"].abs()
)

bootstrap_summary = bootstrap_summary.sort_values(
    "original_abs_coefficient", ascending=False
)

print("Successful bootstrap fits:", len(bootstrap_coefficients))
print("\nBootstrap stability for the 10 largest original coefficients:")
print(
    bootstrap_summary[
        [
            "feature",
            "original_coefficient",
            "bootstrap_mean",
            "bootstrap_std",
            "bootstrap_q025",
            "bootstrap_q975",
            "sign_stability",
        ]
    ].head(10).round(4).to_string(index=False)
)

print(
    "\nNote: These are exploratory training-set stability estimates, "
    "not formal confidence intervals or evidence of causality."
)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are

Successful bootstrap fits: 30

Bootstrap stability for the 10 largest original coefficients:
                  feature  original_coefficient  bootstrap_mean  bootstrap_std  bootstrap_q025  bootstrap_q975  sign_stability
        numeric__RIDAGEYR                0.6065          0.6206         0.0539          0.5420          0.7359             1.0
  categorical__HUQ010_5.0                0.5373          0.4960         0.1286          0.2808          0.7529             1.0
  categorical__DIQ010_1.0                0.4244          0.4276         0.0814          0.3059          0.5735             1.0
  categorical__HUQ010_2.0               -0.3835         -0.3743         0.0807         -0.5102         -0.2074             1.0
  categorical__HUQ010_1.0               -0.3309         -0.3311         0.1216         -0.5164         -0.0887             1.0
  categorical__DIQ010_3.0               -0.3155         -0.3201         0.1297         -0.5217         -0.0501             1.0
numeric__mean_syst

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [41]:
# Cell 40: Summarize bootstrap coefficient stability

bootstrap_stability_summary = pd.DataFrame({
    "total_features": [len(transformed_feature_names)],
    "successful_bootstrap_fits": [len(bootstrap_coefficients)],
    "features_with_stable_sign_90pct": [
        int((bootstrap_summary["sign_stability"] >= 0.90).sum())
    ],
    "features_with_stable_sign_100pct": [
        int((bootstrap_summary["sign_stability"] == 1.0).sum())
    ],
    "features_with_intervals_excluding_zero": [
        int(
            (
                (bootstrap_summary["bootstrap_q025"] > 0)
                | (bootstrap_summary["bootstrap_q975"] < 0)
            ).sum()
        )
    ],
})

print("Bootstrap stability summary:")
print(bootstrap_stability_summary.to_string(index=False))

print("\nFeatures with the least stable coefficient signs:")
print(
    bootstrap_summary.sort_values("sign_stability")[
        [
            "feature",
            "original_coefficient",
            "bootstrap_std",
            "bootstrap_q025",
            "bootstrap_q975",
            "sign_stability",
        ]
    ].head(10).round(4).to_string(index=False)
)

print(
    "\nInterpretation: these summaries describe the observed bootstrap "
    "resamples; they are not formal inferential guarantees."
)

Bootstrap stability summary:
 total_features  successful_bootstrap_fits  features_with_stable_sign_90pct  features_with_stable_sign_100pct  features_with_intervals_excluding_zero
             28                         30                               19                                14                                      17

Features with the least stable coefficient signs:
                  feature  original_coefficient  bootstrap_std  bootstrap_q025  bootstrap_q975  sign_stability
  categorical__PAQ620_1.0                0.0082         0.0382         -0.0701          0.0691          0.5333
  categorical__SMQ020_1.0                0.0027         0.0362         -0.0689          0.0543          0.5333
  categorical__HIQ011_2.0                0.0010         0.0660         -0.1145          0.1469          0.6333
  categorical__PAQ650_2.0               -0.0163         0.0516         -0.1127          0.0786          0.6667
  categorical__PAQ665_2.0                0.0358         0.0435   

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [42]:
# Cell 41: Final integrity checks for evaluation and bootstrap results

assert len(y_test) == 1654
assert len(test_probabilities["Standard Logistic Regression"]) == len(y_test)
assert len(test_probabilities["Class-weighted Logistic Regression"]) == len(y_test)

assert len(bootstrap_coefficients) == BOOTSTRAP_REPEATS
assert bootstrap_coefficients.shape[1] == len(transformed_feature_names)
assert len(bootstrap_summary) == len(transformed_feature_names)

assert final_test_summary["model"].nunique() == 2
assert np.isfinite(bootstrap_coefficients).all()
assert np.isfinite(final_test_summary.select_dtypes(include="number")).all().all()

print("Held-out test participants:", len(y_test))
print("Held-out positive cases:", int(y_test.sum()))
print("Held-out negative cases:", int((y_test == 0).sum()))
print("Models evaluated:", final_test_summary["model"].tolist())
print("Bootstrap fits completed:", len(bootstrap_coefficients))
print("Transformed features:", len(transformed_feature_names))
print("Non-finite bootstrap coefficients:", int((~np.isfinite(bootstrap_coefficients)).sum()))
print("\nNB2 evaluation integrity checks passed.")

Held-out test participants: 1654
Held-out positive cases: 325
Held-out negative cases: 1329
Models evaluated: ['Standard Logistic Regression', 'Class-weighted Logistic Regression']
Bootstrap fits completed: 30
Transformed features: 28
Non-finite bootstrap coefficients: 0

NB2 evaluation integrity checks passed.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [43]:
# Cell 42: Draft the NB2 evaluation results narrative

standard_results = final_test_summary.loc[
    final_test_summary["model"] == "Standard Logistic Regression"
].iloc[0]

weighted_results = final_test_summary.loc[
    final_test_summary["model"] == "Class-weighted Logistic Regression"
].iloc[0]

nb2_results_narrative = f"""
## Model Development and Evaluation — Results Summary

### Evaluation design
The labelled cohort contained {len(y)} participants. A stratified 80/20
train/test split with random_state={SEED} was used. The held-out test set
contained {len(y_test)} participants: {int((y_test == 0).sum())} negative
and {int(y_test.sum())} positive research labels.

### Held-out test performance at threshold 0.50
Standard Logistic Regression achieved accuracy {standard_results['accuracy']:.4f},
precision {standard_results['precision']:.4f}, recall {standard_results['recall']:.4f},
F1 {standard_results['f1']:.4f}, ROC-AUC {standard_results['roc_auc']:.4f},
and average precision {standard_results['average_precision']:.4f}.

Class-weighted Logistic Regression achieved accuracy {weighted_results['accuracy']:.4f},
precision {weighted_results['precision']:.4f}, recall {weighted_results['recall']:.4f},
F1 {weighted_results['f1']:.4f}, ROC-AUC {weighted_results['roc_auc']:.4f},
and average precision {weighted_results['average_precision']:.4f}.

### Probability quality
The standard model had Brier score {standard_results['brier_score']:.4f}
and log loss {standard_results['log_loss']:.4f}; the class-weighted model
had Brier score {weighted_results['brier_score']:.4f}
and log loss {weighted_results['log_loss']:.4f}. Lower values are better
for these metrics.

### Interpretation and limitations
The class-weighted model had higher recall at threshold 0.50, while the
standard model had higher precision and specificity. The models had the
same ROC-AUC on the held-out test set. Model selection therefore depends
on the intended research objective and the consequences of false positives
and false negatives.

The target is a cross-sectional kidney-abnormality research indicator based
on eGFR and urine albumin-to-creatinine ratio, not a confirmed clinical CKD
diagnosis or a prediction of future disease. These results do not establish
clinical readiness, causality, or external validity.
""".strip()

print(nb2_results_narrative)

## Model Development and Evaluation — Results Summary

### Evaluation design
The labelled cohort contained 8267 participants. A stratified 80/20
train/test split with random_state=42 was used. The held-out test set
contained 1654 participants: 1329 negative
and 325 positive research labels.

### Held-out test performance at threshold 0.50
Standard Logistic Regression achieved accuracy 0.8174,
precision 0.6162, recall 0.1877,
F1 0.2877, ROC-AUC 0.7994,
and average precision 0.5084.

Class-weighted Logistic Regression achieved accuracy 0.7152,
precision 0.3866, recall 0.7662,
F1 0.5139, ROC-AUC 0.7994,
and average precision 0.5041.

### Probability quality
The standard model had Brier score 0.1259
and log loss 0.4009; the class-weighted model
had Brier score 0.1867
and log loss 0.5567. Lower values are better
for these metrics.

### Interpretation and limitations
The class-weighted model had higher recall at threshold 0.50, while the
standard model had higher precision and specificity. T

In [48]:
# Cell 43: Verify key NB2 results

expected_test_metrics = {
    "Standard Logistic Regression": {
        "accuracy": 0.8174,
        "recall": 0.1877,
        "roc_auc": 0.7994,
        "brier_score": 0.1259,
    },
    "Class-weighted Logistic Regression": {
        "accuracy": 0.7152,
        "recall": 0.7662,
        "roc_auc": 0.7994,
        "brier_score": 0.1867,
    },
}

for model_name, expected_metrics in expected_test_metrics.items():
    row = final_test_summary.loc[
        final_test_summary["model"] == model_name
    ].iloc[0]

    for metric_name, expected_value in expected_metrics.items():
        assert np.isclose(
            row[metric_name], expected_value, atol=0.0001
        ), (
            f"Mismatch for {model_name}, {metric_name}: "
            f"{row[metric_name]} != {expected_value}"
        )

assert "not a confirmed clinical CKD" in nb2_results_narrative
assert "prediction of future disease" in nb2_results_narrative
assert len(bootstrap_coefficients) == BOOTSTRAP_REPEATS

print("Test metric consistency: passed")
print("Research-target limitations included: passed")
print("Bootstrap fit count:", len(bootstrap_coefficients))
print("NB2 results narrative available:", bool(nb2_results_narrative))
print("\nNB2 results consistency checks passed.")

Test metric consistency: passed
Research-target limitations included: passed
Bootstrap fit count: 30
NB2 results narrative available: True

NB2 results consistency checks passed.


In [49]:
# Cell 44: Export aggregate held-out test results

from pathlib import Path

results_dir = Path("/content/results")
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "nb2_heldout_test_metrics.csv"

final_test_summary.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(final_test_summary))
print("Models:", final_test_summary["model"].tolist())
print("Columns:", final_test_summary.columns.tolist())

Saved: /content/results/nb2_heldout_test_metrics.csv
Rows: 2
Models: ['Standard Logistic Regression', 'Class-weighted Logistic Regression']
Columns: ['model', 'threshold', 'accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc', 'average_precision', 'brier_score', 'log_loss']


# Model Development and Evaluation — Results Summary

## Evaluation design

The labelled cohort contained 8,267 participants. A stratified 80/20 train/test split (random_state=42) was used. The held-out test set contained 1,654 participants: 1,329 negative and 325 positive research labels.

Held-out test performance at threshold 0.50

Metric	Standard Logistic Regression	Class-weighted Logistic Regression
Accuracy	0.8174	0.7152
Precision	0.6162	0.3866
Recall	0.1877	0.7662
F1-score	0.2877	0.5139
Specificity	0.9714	0.7028
ROC-AUC	0.7994	0.7994
Average precision	0.5084	0.5041
Brier score	0.1259	0.1867
Log loss	0.4009	0.5567

## Interpretation

At the default threshold of 0.50, the class-weighted model achieved higher recall and F1-score, while the standard model achieved higher precision, specificity, and accuracy. Both models had the same held-out ROC-AUC. The standard model also had lower Brier score and log loss, indicating better probability quality according to these metrics.

The choice between models depends on the research objective and the relative consequences of false positives and false negatives. The reported threshold comparison is descriptive; a decision threshold should be selected using training-only validation data and a clearly stated objective, not by optimising performance on the held-out test set.

## Interpretability and stability

The weighted model’s coefficients describe associations learned by the fitted model, not causal effects. Bootstrap analysis used 30 training-set refits to explore coefficient stability. These results are exploratory and do not constitute formal inferential guarantees.

## Limitations

The target is a cross-sectional kidney-abnormality research indicator based on eGFR and urine albumin-to-creatinine ratio. It is not a confirmed clinical CKD diagnosis and does not predict future disease. A single cross-sectional assessment cannot establish the persistence required for a clinical CKD diagnosis. These analyses do not establish clinical readiness, causality, or external validity.